In [4]:
import pandas as pd 
import numpy as np 

In [5]:
df_orders = pd.read_csv('orders.csv')
df_orders.head()

,order_id,date,time
0,1,2015-01-01,11:38:36
1,2,2015-01-01,11:57:40
2,3,2015-01-01,12:12:28
3,4,2015-01-01,12:16:31
4,5,2015-01-01,12:21:30


In [6]:
df_order_details = pd.read_csv('order_details.csv')
df_order_details.head()

,order_details_id,order_id,pizza_id,quantity
0,1,1,hawaiian_m,1
1,2,2,classic_dlx_m,1
2,3,2,five_cheese_l,1
3,4,2,ital_supr_l,1
4,5,2,mexicana_m,1


In [7]:
df_pizzas = pd.read_csv('pizzas.csv')
df_pizzas.head()

,pizza_id,pizza_type_id,size,price
0,bbq_ckn_s,bbq_ckn,S,12.75
1,bbq_ckn_m,bbq_ckn,M,16.75
2,bbq_ckn_l,bbq_ckn,L,20.75
3,cali_ckn_s,cali_ckn,S,12.75
4,cali_ckn_m,cali_ckn,M,16.75


In [8]:
df_pizza_type = pd.read_csv('pizza_types.csv', encoding='windows-1252')
df_pizza_type.head()

,pizza_type_id,name,category,ingredients
0,bbq_ckn,The Barbecue Chicken Pizza,Chicken,"Barbecued Chicken, Red Peppers, Green Peppers,..."
1,cali_ckn,The California Chicken Pizza,Chicken,"Chicken, Artichoke, Spinach, Garlic, Jalapeno ..."
2,ckn_alfredo,The Chicken Alfredo Pizza,Chicken,"Chicken, Red Onions, Red Peppers, Mushrooms, A..."
3,ckn_pesto,The Chicken Pesto Pizza,Chicken,"Chicken, Tomatoes, Red Peppers, Spinach, Garli..."
4,southw_ckn,The Southwest Chicken Pizza,Chicken,"Chicken, Tomatoes, Red Peppers, Red Onions, Ja..."


In [9]:
# #cleaning orders Table 
# df_orders.isnull().values.any()
# df_orders.info()
df_orders.count() # count rows for each column 

order_id    21350
date        21350
time        21350
dtype: int64

In [10]:
df_orders.isnull().sum() #checking nulls count for each column 

order_id    0
date        0
time        0
dtype: int64

In [12]:
# Extract hour (assuming 'time' is already datetime)
df_orders['hour'] = df_orders['time'].dt.hour

# Count orders per hour
hourly_sales = df_orders.groupby('hour').size()

# Peak hours (top 3)
hourly_sales.nlargest(3)   

AttributeError: Can only use .dt accessor with datetimelike values

In [ ]:
df_orders.tail()

In [13]:
df_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 21350 entries, 0 to 21349
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   order_id  21350 non-null  int64
 1   date      21350 non-null  str  
 2   time      21350 non-null  str  
dtypes: int64(1), str(2)
memory usage: 875.8 KB


In [ ]:
df_orders['date'] = pd.to_datetime(df_orders['date'])

In [ ]:
df_orders.columns.tolist()

In [ ]:

# 1. Merge the three DataFrames
# order_details + pizzas to get price
df_merged = df_order_details.merge(df_pizzas[['pizza_id', 'price']], on='pizza_id', how='left')

# add orders to get order date
df_merged = df_merged.merge(df_orders[['order_id', 'date']], on='order_id', how='left')

# 2. Calculate revenue per line item
df_merged['revenue'] = df_merged['quantity'] * df_merged['price']

# 3. Ensure 'date' is in datetime format
df_merged['date'] = pd.to_datetime(df_merged['date'])

# 4. Calculate total revenue for each unique date
daily_revenue = df_merged.groupby('date', as_index=False)['revenue'].sum()

# 5. Extract month, day of month, and day of week for pattern analysis
daily_revenue['year_month'] = daily_revenue['date'].dt.to_period('M')
daily_revenue['day_name'] = daily_revenue['date'].dt.day_name()
daily_revenue['day_of_month'] = daily_revenue['date'].dt.day

# ---------------------------------------------------------
# Scenario A: Top revenue date for EVERY month in the year
# ---------------------------------------------------------
peak_dates_per_month = daily_revenue.loc[
    daily_revenue.groupby('year_month')['revenue'].idxmax()
].reset_index(drop=True)

print("Peak Revenue Date for Each Month:")
print(peak_dates_per_month[['year_month', 'date', 'day_name', 'day_of_month', 'revenue']])

# ---------------------------------------------------------
# Scenario B: The single highest revenue date across the entire year
# ---------------------------------------------------------
all_time_peak_date = daily_revenue.loc[daily_revenue['revenue'].idxmax()]

print("\nSingle Highest Revenue Date of the Year:")
print(all_time_peak_date)

In [ ]:
df_merged.head()


In [ ]:

# 1. Check the dataset dimensions
print("Rows and columns:", df_merged.shape)

# 2. Inspect column names and data types
df_merged.info()

# 3. Check missing values
print(df_merged.isnull().sum())

# 4. Check duplicate rows
print("Duplicate rows:", df_merged.duplicated().sum())

# 5. Get summary statistics
display(df_merged.describe(include="all"))


In [ ]:
print("Unique order-detail IDs:", df_merged["order_details_id"].nunique())
print("Unique order IDs:", df_merged["order_id"].nunique())
print("Unique pizza IDs:", df_merged["pizza_id"].nunique())

In [ ]:
print("Quantity <= 0:", (df_merged["quantity"] <= 0).sum())
print("Price <= 0:", (df_merged["price"] <= 0).sum())
print("Revenue < 0:", (df_merged["revenue"] < 0).sum())

In [ ]:
print("First date:", df_merged["date"].min())
print("Last date:", df_merged["date"].max())
print("Unique dates:", df_merged["date"].nunique())

In [ ]:
df_merged.head()

In [ ]:
#Creating a unified table with all cleaned and analysis ready data 
df_master = df_order_details.merge(df_orders,on='order_id', how='left')
df_master = df_master.merge(df_pizzas, on='pizza_id', how='left')
df_master = df_master.merge(df_pizza_type, on='pizza_type_id', how='left')
df_master.head() #head of the final data

In [ ]:
df_master.rename(
    columns={
        'date': 'order_date',
        'time': 'order_time',
        'name': 'pizza_name',
        'category': 'pizza_category',
    },
    inplace=True,
)

# 2. Financial calculation
df_master['revenue'] = (df_master['quantity'] * df_master['price']).round(2)

# 3. Date & Calendar Dimensions
df_master['order_date'] = pd.to_datetime(df_master['order_date'])
df_master['month_name'] = df_master['order_date'].dt.month_name()
df_master['month_number'] = df_master['order_date'].dt.month  # Used in Power BI to sort 'month_name'
df_master['day_name'] = df_master['order_date'].dt.day_name()
df_master['day_of_week'] = df_master['order_date'].dt.dayofweek  # Used in Power BI to sort 'day_name' (0=Mon, 6=Sun)
df_master['day_type'] = np.where(
    df_master['day_of_week'] >= 5, 'Weekend', 'Weekday'
)
df_master['quarter'] = 'Q' + df_master['order_date'].dt.quarter.astype(str)

# 4. Time & Peak Hour Dimensions
# Using str.split(':') extracts the hour directly and avoids date format warnings
df_master['order_hour'] = df_master['order_time'].astype(str).str.split(':').str[0].astype(int)

# Create meal time / rush periods
conditions = [
    (df_master['order_hour'] >= 11) & (df_master['order_hour'] < 15),
    (df_master['order_hour'] >= 15) & (df_master['order_hour'] < 18),
    (df_master['order_hour'] >= 18) & (df_master['order_hour'] < 22),
]
choices = ['Lunch Rush', 'Afternoon Lull', 'Dinner Rush']
df_master['meal_period'] = np.select(
    conditions, choices, default='Late Night / Other'
)

# 5. Expand Pizza Size acronyms to clean display names
size_map = {
    'S': 'Small',
    'M': 'Medium',
    'L': 'Large',
    'XL': 'X-Large',
    'XXL': 'XX-Large',
}
clean_size = df_master['size'].astype(str).str.strip().str.upper()
df_master['size_full_name'] = clean_size.map(size_map).fillna(df_master['size'])

In [ ]:
print("Master Shape:", df_master.shape)
print("\n--- Missing Values Check ---")
print(df_master.isna().sum()[df_master.isna().sum() > 0])  # Should ideally be empty

print("\n--- Executive Summary KPIs ---")
print(f"Total Revenue:       ${df_master['revenue'].sum():,.2f}")
print(f"Total Pizzas Sold:   {df_master['quantity'].sum():,}")
print(f"Total Unique Orders: {df_master['order_id'].nunique():,}")
print(f"Average Order Value: ${df_master['revenue'].sum() / df_master['order_id'].nunique():,.2f}")

print("\n--- Sample Output Preview ---")
df_master[[
    'order_id', 'order_date', 'order_hour', 'meal_period',
    'pizza_name', 'pizza_category', 'size_full_name',
    'quantity', 'price', 'revenue'
]].head(5)

In [ ]:
# Create a 12-hour AM/PM display column from order_hour
df_master['order_hour_am_pm'] = df_master['order_hour'].apply(
    lambda h: f"{12 if h % 12 == 0 else h % 12} {'AM' if h < 12 else 'PM'}"
)

In [ ]:
print("\n--- Sample Output Preview ---")
df_master[[
    'order_id', 'order_date', 'order_hour', 'meal_period',
    'pizza_name', 'pizza_category', 'size_full_name',
    'quantity', 'price', 'revenue','order_hour_am_pm'
]].head(5)

In [ ]:
df_master.to_excel('master.xlsx', index=False)

In [ ]:
df_master.to_csv('master.csv',index=False)